# Deshimula Company Review Scraper

Collects company reviews from [Deshimula](https://deshimula.com) (a Bangladeshi company-review site like Glassdoor) into a CSV.

## How it works

1. **Companies** are defined as `(slug, display_name)` tuples. Their internal `companyId` is resolved from the company page HTML.
2. **Vibes** are the review sentiment buckets exposed by the site's URL param: `1` = Positive, `2` = Negative, `3` = Mixed. The vibe is therefore known without any NLP.
3. **Pagination** is done via `/stories/{page}?companyId={id}&Vibe={v}`. Note: `SearchTerm`-based pagination is broken (page 2+ returns nothing), so we resolve `companyId` and use it instead.
4. **Listing cards** provide `title`, `role`, `date`, `upvote`, `downvote`, `comment_count` (and a truncated `content`).
5. **Detail pages** (`/story/{id}`) provide the full `content` (the list view truncates it behind "Read More").
6. Each review gets an incremental `id`.

In [ ]:
import csv
import re
import time
from pathlib import Path

import requests
from bs4 import BeautifulSoup

BASE_URL = "https://deshimula.com"

VIBES = {
    1: "Positive",
    2: "Negative",
    3: "Mixed",
}

# (company slug, display name used in the CSV)
COMPANIES = [
    ("brain-station-23", "Brain Station 23"),
    ("optimizely", "Optimizely"),
]

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
}

OUTPUT = Path.cwd() / "deshimula_reviews.csv"
SLEEP = 1.0

In [ ]:
def new_session() -> requests.Session:
    session = requests.Session()
    session.headers.update(HEADERS)
    return session


def resolve_company_id(session: requests.Session, slug: str) -> str:
    """The companyId appears in links to /stories/1?companyId=... on the company page."""
    r = session.get(f"{BASE_URL}/companies/{slug}", timeout=20)
    r.raise_for_status()
    m = re.search(r"/stories/1\?companyId=([0-9a-f]{24})", r.text)
    if not m:
        raise RuntimeError(f"Could not find companyId for {slug}")
    return m.group(1)

In [ ]:
def parse_card(card) -> dict:
    """Extract fields from one listing-card container."""
    link = next(
        a
        for a in card.find_all("a", href=True)
        if re.match(r"/story/[0-9a-f]{24}", a["href"]) and "Read" in a.get_text()
    )
    story_id = link["href"].split("/")[-1]

    h2 = card.find("h2")
    title = h2.get_text(" ", strip=True) if h2 else ""

    meta = card.find("div", class_=re.compile("mt-0.5"))
    role = date = ""
    if meta:
        spans = [s.get_text(strip=True) for s in meta.find_all("span")]
        spans = [s for s in spans if s]
        if len(spans) >= 3:
            role, _, date = spans[0], spans[1], spans[2]

    content_div = card.find("div", class_=re.compile("text-slate-800"))
    snippet = content_div.get_text(" ", strip=True) if content_div else ""

    votes_cont = None
    for div in card.find_all("div"):
        if div.find(string=re.compile("Upvotes")):
            votes_cont = div
            break
    nums = []
    if votes_cont:
        nums = [
            s.get_text(strip=True)
            for s in votes_cont.find_all("span")
            if s.get_text(strip=True).isdigit()
        ][-3:]  # last 3 numbers = upvote, downvote, comment_count
    nums = (nums + ["0"] * 3)[:3]

    return {
        "story_id": story_id,
        "title": title,
        "role": role,
        "date": date,
        "content": snippet,
        "upvote": int(nums[0]),
        "downvote": int(nums[1]),
        "comment_count": int(nums[2]),
    }

In [ ]:
def scrape_listing_page(session: requests.Session, company_id: str, vibe: int, page: int) -> list[dict]:
    url = f"{BASE_URL}/stories/{page}?companyId={company_id}&Vibe={vibe}"
    r = session.get(url, timeout=20)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")

    rows = []
    for link in soup.find_all("a", href=True):
        if re.match(r"/story/[0-9a-f]{24}", link["href"]) and "Read" in link.get_text():
            card = link.find_parent("div", class_="container")
            if card is None:
                continue
            rows.append(parse_card(card))
    return rows

In [ ]:
def fetch_full_content(session: requests.Session, story_id: str) -> str:
    """The listing card truncates content behind 'Read More'; grab the full text from /story/{id}."""
    r = session.get(f"{BASE_URL}/story/{story_id}", timeout=20)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")
    content = soup.find("div", class_="story-content")
    if content:
        return content.get_text("\n", strip=True)
    return ""

In [ ]:
def scrape_all() -> list[dict]:
    session = new_session()
    reviews = []

    for slug, display_name in COMPANIES:
        company_id = resolve_company_id(session, slug)
        print(f"[{display_name}] companyId={company_id}")

        for vibe_val, vibe_label in VIBES.items():
            page = 1
            while True:
                rows = scrape_listing_page(session, company_id, vibe_val, page)
                if not rows:
                    break
                for row in rows:
                    time.sleep(SLEEP)
                    full = fetch_full_content(session, row["story_id"])
                    if full:
                        row["content"] = full
                    reviews.append(
                        {
                            "id": len(reviews) + 1,
                            "company_name": display_name,
                            "developer_role": row["role"],
                            "date": row["date"],
                            "title": row["title"],
                            "content": row["content"],
                            "upvote": row["upvote"],
                            "downvote": row["downvote"],
                            "comment_count": row["comment_count"],
                            "vibe": vibe_label,
                        }
                    )
                print(f"  {vibe_label}: {len(reviews)} total after page {page}")
                page += 1
                time.sleep(SLEEP)

    return reviews


def dedupe(reviews: list[dict]) -> list[dict]:
    seen, out = set(), []
    for r in reviews:
        key = (r["company_name"], r["title"], r["content"])
        if key in seen:
            continue
        seen.add(key)
        out.append(r)
    return out

In [ ]:
FIELDS = [
    "id",
    "company_name",
    "developer_role",
    "date",
    "title",
    "content",
    "upvote",
    "downvote",
    "comment_count",
    "vibe",
]

reviews = scrape_all()
reviews = dedupe(reviews)
for i, r in enumerate(reviews, start=1):
    r["id"] = i

with OUTPUT.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=FIELDS)
    writer.writeheader()
    writer.writerows(reviews)

print(f"Saved {len(reviews)} reviews -> {OUTPUT}")